# Software Career Navigator — Salary Prediction and Field Recommender

**End-to-End AI Project — Software Persona Internship**

**Problem:** Predict the expected monthly salary (USD and TL) of a software developer based on years of experience, field of expertise, country, education level, company size, work arrangement, industry, age, title (IC/PM), AI tool usage, and number of known languages; and recommend the most suitable of 9 software fields based on personal preferences.

**Dataset:** Stack Overflow Developer Survey 2025 (Kaggle) — Responses from 49,123 developers

**Scope:** Data Cleaning → Exploratory Data Analysis → Model Comparison → Hyperparameter Optimization → Evaluation → Field Recommender Mini App → Exporting Model (for deployment)

## Installation

In [ ]:
# --- Load required libraries ---
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

try:
    import xgboost as xgb
except ImportError:
    %pip install xgboost --quiet
    import xgboost as xgb

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Installation tamamlandı. Rastgelelik sabiti:', RANDOM_STATE)

## Phase 2: Data Collection and Preparation

The notebook first looks for files next to it (reads locally or from Colab uploads). If not found, it tries to automatically download via Kaggle API. This prevents manual re-uploads when Colab sessions restart.

In [ ]:
# --- Prepare Kaggle Data ---
# Note: Since Kaggle download requires an account/API key, automatic download
# might not work in every environment. We first check possible locations;
# if not found, we try downloading with Kaggle API/CLI (Kaggle API key
# must be defined in the environment: ~/.kaggle/kaggle.json).

DOSYA_ADI = 'survey_results_public.csv'

adaylar = [
    Path.cwd() / DOSYA_ADI,
    Path.cwd() / 'dataset' / DOSYA_ADI,
    Path('/content') / DOSYA_ADI,
    Path('/content/dataset') / DOSYA_ADI,
]

veri_yolu = next((yol for yol in adaylar if yol.exists()), None)

if veri_yolu is None:
    print('File not found locally, attempting download via Kaggle API...')
    print("(If Kaggle API key is not set, manually upload the file to Colab")
    print(' and re-run this cell.)')
    try:
        import kaggle
        hedef_klasor = Path('/content/dataset') if Path('/content').exists() else Path.cwd() / 'dataset'
        hedef_klasor.mkdir(parents=True, exist_ok=True)
        kaggle.api.dataset_download_files(
            'edoardogalli/stack-overflow-annual-developer-survey-2025',
            path=str(hedef_klasor), unzip=True
        )
        veri_yolu = hedef_klasor / DOSYA_ADI
    except Exception as hata:
        print('Automatic download failed:', hata)
        print("Please manually upload the survey_results_public.csv file to Colab.")

print('Data used:', veri_yolu)

df = pd.read_csv(veri_yolu)
print('Raw data shape:', df.shape)

In [ ]:
# --- Select columns for the project (extended feature set) ---
# WorkExp: professional work experience (years)
# DevType: developer role/expertise
# Country: country
# EdLevel: education level
# RemoteWork: work arrangement (remote/hybrid/office)
# OrgSize: company size
# Industry: industry
# Employment: employment type
# Age: age range
# ICorPM: manager or individual contributor
# AISelect: AI tool usage frequency
# LanguageHaveWorkedWith: known programming languages (separated by ';') -> will be converted to count
# ConvertedCompYearly: annual salary (converted to USD) — source of our target variable

cols_extended = ['WorkExp', 'DevType', 'Country', 'EdLevel',
                 'RemoteWork', 'OrgSize', 'Industry', 'Employment', 'Age',
                 'ICorPM', 'AISelect', 'LanguageHaveWorkedWith',
                 'ConvertedCompYearly']

df_raw = df[cols_extended].copy()
df_raw = df_raw.dropna(subset=cols_extended)

# --- Filter unrealistic outliers ---
df_raw = df_raw[(df_raw['ConvertedCompYearly'] >= 1000) & (df_raw['ConvertedCompYearly'] <= 500000)]
df_raw = df_raw[df_raw['WorkExp'] <= 50]

# --- Derive language count ---
df_raw['DilSayisi'] = df_raw['LanguageHaveWorkedWith'].apply(lambda x: len(str(x).split(';')))

# --- Derive monthly salary column ---
df_clean = df_raw.copy()
df_clean['MonthlySalaryUSD'] = df_clean['ConvertedCompYearly'] / 12

print('Cleaned data shape:', df_clean.shape)

### Data Dictionary

| Column | Type | Meaning |
|---|---|---|
| WorkExp | Numeric | Professional work experience (years) |
| DevType | Categorical | Developer role/expertise |
| Country | Categorical | Country of residence |
| EdLevel | Categorical | Education level |
| RemoteWork | Categorical | Work arrangement (remote/hybrid/office) |
| OrgSize | Categorical | Company size |
| Industry | Categorical | Industry |
| Employment | Categorical | Employment type |
| Age | Categorical | Age range |
| ICorPM | Categorical | Individual contributor or manager |
| AISelect | Categorical | AI tool usage frequency |
| DilSayisi | Numeric | Number of known programming languages (derived from LanguageHaveWorkedWith) |
| MonthlySalaryUSD | Numeric | Target variable — monthly salary (USD) |

**Cleaning Steps:** Rows with missing data were dropped; salary outliers outside 1,000-500,000 USD/year and rows claiming over 50 years of experience were filtered. Result: 18,223 clean rows.

## Phase 3: Exploratory Data Analysis (EDA) and Visualization

In [ ]:
# --- General summary ---
print(df_clean.describe())
print(df_clean['DevType'].value_counts().head(10))
print(df_clean['Country'].value_counts().head(10))

In [ ]:
# --- Chart 1: Average monthly salary by years of experience ---
# We use an average line by grouping into 2-year buckets to clearly show the trend
# (raw scatter plot was avoided due to clustering of too many points)

df_clean['ExpGroup'] = (df_clean['WorkExp'] // 2) * 2
avg_by_exp = df_clean.groupby('ExpGroup')['MonthlySalaryUSD'].mean()

plt.figure(figsize=(10, 6))
plt.plot(avg_by_exp.index, avg_by_exp.values, marker='o', color='steelblue')
plt.title('Average Monthly Salary by Years of Experience (USD)')
plt.xlabel('Professional Experience (Years)')
plt.ylabel('Average Monthly Salary (USD)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Chart 2: Average monthly salary by experience for DevOps engineers ---

df_devops = df_clean[df_clean['DevType'] == 'DevOps engineer or professional'].copy()
print('Number of DevOps engineers:', df_devops.shape[0])

avg_devops = df_devops.groupby('ExpGroup')['MonthlySalaryUSD'].mean()

plt.figure(figsize=(10, 6))
plt.plot(avg_devops.index, avg_devops.values, marker='o', color='darkorange')
plt.title('DevOps Mühendislerinde Deneyime Göre Average Monthly Salary (USD)')
plt.xlabel('Professional Experience (Years)')
plt.ylabel('Average Monthly Salary (USD)')
plt.grid(alpha=0.3)
plt.show()

### Exploratory Findings

1. There is a rapid and clear increase in salary during the first 10 years of experience.
2. The increase continues between 10-30 years, but at a slower pace.
3. Volatility increases above 30 years of experience — this might be statistical noise due to the low number of samples at those levels.
4. A similar trend exists for DevOps engineers (672 people): rapid growth in early years, followed by a slowing rise.
5. The most represented roles and countries in the general sample show that the dataset is heavily US and Western Europe skewed — this means the model might produce less reliable predictions for some regions.

## Phase 4: Model Building, Comparison and Optimization

In [ ]:
# --- Split features (X) and target (y) ---
X = df_clean[['WorkExp', 'DevType', 'Country', 'EdLevel',
              'RemoteWork', 'OrgSize', 'Industry', 'Employment', 'Age',
              'ICorPM', 'AISelect', 'DilSayisi']]
y = df_clean['MonthlySalaryUSD']

categorical_cols = ['DevType', 'Country', 'EdLevel', 'RemoteWork', 'OrgSize',
                     'Industry', 'Employment', 'Age', 'ICorPM', 'AISelect']
# WorkExp and LanguageCount remain numeric (remainder='passthrough')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)

### Model Comparison

We test and compare the performances of four different regression models: starting with a simple linear model (LinearRegression) and moving to tree-based models (RandomForest, GradientBoosting, XGBoost) that can capture non-linear relationships.

In [ ]:
sonuclar = {}

# --- Model 1: LinearRegression ---
model_lr = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)
sonuclar['LinearRegression'] = (r2_score(y_test, pred_lr), mean_absolute_error(y_test, pred_lr))

# --- Model 2: RandomForestRegressor ---
model_rf = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(
    n_estimators=300, max_depth=None, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
))])
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)
sonuclar['RandomForest'] = (r2_score(y_test, pred_rf), mean_absolute_error(y_test, pred_rf))

# --- Model 3: GradientBoostingRegressor ---
model_gb = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE
))])
model_gb.fit(X_train, y_train)
pred_gb = model_gb.predict(X_test)
sonuclar['GradientBoosting'] = (r2_score(y_test, pred_gb), mean_absolute_error(y_test, pred_gb))

print('First 3 models trained:')
for isim, (r2, mae) in sonuclar.items():
    print(f'  {isim}: R²={r2:.3f}, MAE={mae:.2f} USD/ay')

### Hyperparameter Optimization (XGBoost + GridSearch)

We optimize XGBoost with GridSearch to get the best result — this tries multiple hyperparameter combinations with cross-validation and selects the best one.

In [ ]:
# --- Model 4: XGBoost + GridSearch ---
model_xgb_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid = {
    'regressor__n_estimators': [200, 300, 400],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_depth': [3, 4, 5],
}

grid_search = GridSearchCV(
    model_xgb_base, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)

print('GridSearch starting, this might take a few minutes...')
grid_search.fit(X_train, y_train)

print('\nBest parameters:', grid_search.best_params_)
print('Best cross-validation R²:', grid_search.best_score_)

# --- Final model: Best model found by GridSearch ---
model = grid_search.best_estimator_
y_pred = model.predict(X_test)

sonuclar['XGBoost (optimize)'] = (r2_score(y_test, y_pred), mean_absolute_error(y_test, y_pred))

print('\n=== COMPARISON OF ALL MODELS ===')
for isim, (r2, mae) in sonuclar.items():
    print(f'  {isim}: R²={r2:.3f}, MAE={mae:.2f} USD/ay')

print('\n>>> Optimized XGBoost was selected as the final model. <<<')

**Model Selection Rationale:** Four models were compared: simple LinearRegression provided a baseline; RandomForest and GradientBoosting tried capturing non-linear relationships; optimized XGBoost with GridSearch gave the best result (highest R², lowest MAE). Therefore, XGBoost was chosen as the final model. Deep learning (neural networks) was intentionally avoided because: (1) dataset size (~18,000 rows) is relatively small to feed a neural network, (2) since the data is tabular, tree-based models generally perform equally or superior to neural networks in literature, with less data and shorter training times.

## Phase 5: Prediction and Evaluation

In [ ]:
print(f'Final Model (XGBoost) — R²: {r2_score(y_test, y_pred):.3f}')
print(f'Final Model (XGBoost) — MAE: {mean_absolute_error(y_test, y_pred):.2f} USD/ay')

In [ ]:
# --- Generate predictions for 3 new sample profiles ---
USD_TO_TRY = 40  # fixed exchange rate assumption (noted in report)

yeni_ornekler = pd.DataFrame([
    {'WorkExp': 2, 'DevType': 'DevOps engineer or professional', 'Country': 'Turkey',
     'EdLevel': 'Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)',
     'RemoteWork': 'Remote', 'OrgSize': '20 to 99 employees',
     'Industry': 'Software Development', 'Employment': 'Employed', 'Age': '25-34 years old',
     'ICorPM': 'Individual contributor', 'AISelect': 'Yes, I use AI tools weekly', 'DilSayisi': 4},

    {'WorkExp': 8, 'DevType': 'DevOps engineer or professional', 'Country': 'Turkey',
     'EdLevel': 'Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)',
     'RemoteWork': 'Remote', 'OrgSize': '100 to 499 employees',
     'Industry': 'Software Development', 'Employment': 'Employed', 'Age': '25-34 years old',
     'ICorPM': 'Individual contributor', 'AISelect': 'Yes, I use AI tools daily', 'DilSayisi': 7},

    {'WorkExp': 15, 'DevType': 'Developer, back-end', 'Country': 'United States of America',
     'EdLevel': 'Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)',
     'RemoteWork': 'Hybrid (some in-person, leans heavy to flexibility)', 'OrgSize': '1,000 to 4,999 employees',
     'Industry': 'Software Development', 'Employment': 'Employed', 'Age': '35-44 years old',
     'ICorPM': 'People manager', 'AISelect': 'Yes, I use AI tools daily', 'DilSayisi': 9},
])

tahminler = model.predict(yeni_ornekler)

for i, tahmin in enumerate(tahminler):
    tahmin_tl = tahmin * USD_TO_TRY
    print(f'Sample {i+1}: Estimated Monthly Salary = {tahmin:.2f} USD (~{tahmin_tl:,.0f} TL)')

### Evaluation Note

The final model (XGBoost, optimized) can explain a significant portion of the variance in salary and provided a distinct improvement over the initial simple model (R²=0.468). This result is realistic: many factors affecting salary (specific tech skills, negotiation power, company's financial status, etc.) are not present in the dataset. The model doesn't carry an overfitting risk because performance was measured on an unseen test set and GridSearch selected hyperparameters using cross-validation.

### Analyze Model Signals (interpretation via GradientBoosting)

Since interpreting XGBoost's raw coefficients isn't as straightforward as linear models, we examine GradientBoosting's feature importance to see which features pull the salary up/down.

In [ ]:
# --- Visualize GradientBoosting feature importances ---
ozellik_adlari = model_gb.named_steps['preprocessor'].get_feature_names_out()
onemler = pd.Series(
    model_gb.named_steps['regressor'].feature_importances_,
    index=ozellik_adlari
)

onemler.index = (
    onemler.index.str.replace('cat__', '', regex=False)
    .str.replace('remainder__', '', regex=False)
)

en_onemli_10 = onemler.nlargest(10).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(en_onemli_10.index, en_onemli_10.values, color='#1F3564')
plt.title('Top 10 Most Impactful Features for Salary Prediction')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

en_onemli_10.round(4)

**Note:** Feature importance indicates how frequently and effectively a feature is used in the model's predictions; it doesn't imply causality on its own (e.g. "being in the US" alone doesn't increase salary, it's related to market conditions). Nevertheless, this chart transparently shows which factors the model weights more heavily.

## Phase 6: Smart Prediction Tool — Extended Field Recommender

Recommends the most suitable of 9 software fields via a 15-question, 4-option quiz. For fair comparison among fields, scores are normalized against the maximum possible score for each field (converted to percentages) — otherwise, some fields (e.g. Game Development) structurally had lower maximum scores than others (e.g. Data Science) and could fall behind unfairly.

Based on the recommended field, the user is also presented with the link to the official learning roadmap on **roadmap.sh** — thus, the app doesn't just say "this field suits you", but provides a concrete next step saying "here's the step-by-step map of what you need to learn in this field".

In [ ]:
def alan_oner():
    """Recommends the most suitable of 9 software fields via a 15-question, 4-option quiz.
    Scores are normalized against the maximum possible score
    for fair comparison.
    Provides a learning roadmap link via roadmap.sh based on the recommended field."""

    puanlar = {
        'Backend': 0, 'Frontend': 0, 'Mobile': 0, 'DevOps/Cloud': 0,
        'Data Science/AI-ML': 0, 'QA/Test': 0, 'Cybersecurity': 0,
        'Game Development': 0, 'Embedded/IoT': 0,
    }

    # --- Official roadmap link on roadmap.sh for each field ---
    # Note: There is no single official page for "Mobile" and "Embedded/IoT" on roadmap.sh;
    # Android roadmap provided for Mobile, closest general resource provided for Embedded/IoT.
    roadmap_linkleri = {
        'Backend': 'https://roadmap.sh/backend',
        'Frontend': 'https://roadmap.sh/frontend',
        'Mobile': 'https://roadmap.sh/android',
        'DevOps/Cloud': 'https://roadmap.sh/devops',
        'Data Science/AI-ML': 'https://roadmap.sh/ai-data-scientist',
        'QA/Test': 'https://roadmap.sh/qa',
        'Cybersecurity': 'https://roadmap.sh/cyber-security',
        'Game Development': 'https://roadmap.sh/game-developer',
        'Embedded/IoT': 'https://roadmap.sh/computer-science',
    }

    sorular = [
        {'soru': "1) What would you most like to spend time doing in a project?",
         'secenekler': {
            '1': ("Building server-side business logic and APIs", {'Backend': 3, 'DevOps/Cloud': 1}),
            '2': ("Designing the screens the user sees", {'Frontend': 3, 'Mobile': 1}),
            '3': ("Building models from data and generating predictions", {'Data Science/AI-ML': 3}),
            '4': ("Coding the mechanics of a game", {'Game Development': 3}),
         }},
        {'soru': "2) Which environment is more appealing for you to work in?",
         'secenekler': {
            '1': ("Cloud infrastructure, servers, automation pipelines", {'DevOps/Cloud': 3, 'Backend': 1}),
            '2': ("Phone/tablet application development environment", {'Mobile': 3}),
            '3': ("Systems integrated with physical hardware and sensors", {'Embedded/IoT': 3}),
            '4': ("Environment for testing/protecting systems against attacks", {'Cybersecurity': 3}),
         }},
        {'soru': "3) What is your first instinct when you encounter a software bug?",
         'secenekler': {
            '1': ("Systematically write test scenarios to catch the bug", {'QA/Test': 3}),
            '2': ("Follow code logic line by line to find the root cause", {'Backend': 2, 'DevOps/Cloud': 1}),
            '3': ("Look at where it appears wrong in the user interface", {'Frontend': 2, 'Mobile': 1}),
            '4': ("Check if there is a security vulnerability", {'Cybersecurity': 2, 'QA/Test': 1}),
         }},
        {'soru': "4) Which topic makes you feel more curious?",
         'secenekler': {
            '1': ("AI and machine learning models", {'Data Science/AI-ML': 3}),
            '2': ("Game engines and interactive experiences", {'Game Development': 3}),
            '3': ("IoT devices and embedded systems", {'Embedded/IoT': 3}),
            '4': ("Cloud architecture and scalable systems", {'DevOps/Cloud': 3}),
         }},
        {'soru': "5) What do you care most about when developing a product?",
         'secenekler': {
            '1': ("Code performance and database efficiency", {'Backend': 3}),
            '2': ("Fluid and aesthetic user experience", {'Frontend': 3, 'Mobile': 1}),
            '3': ("Testing every scenario to work flawlessly", {'QA/Test': 3}),
            '4': ("Keeping data secure and confidential", {'Cybersecurity': 3}),
         }},
        {'soru': "6) Which working style suits you best?",
         'secenekler': {
            '1': ("Progressing by analyzing deeply on my own", {'Data Science/AI-ML': 2, 'Embedded/IoT': 1}),
            '2': ("Progressing fast with a continuous test-develop cycle", {'Game Development': 2, 'Frontend': 1}),
            '3': ("Progressing by monitoring and automating systems", {'DevOps/Cloud': 2, 'Backend': 1}),
            '4': ("Writing detailed scenarios and verifying step-by-step", {'QA/Test': 2, 'Cybersecurity': 1}),
         }},
        {'soru': "7) Which of the following excites you the most?",
         'secenekler': {
            '1': ("A mobile application reaching millions of users", {'Mobile': 3}),
            '2': ("A model generating accurate predictions", {'Data Science/AI-ML': 3}),
            '3': ("A system being resilient against attacks", {'Cybersecurity': 3}),
            '4': ("A game being smooth and entertaining", {'Game Development': 3}),
         }},
        {'soru': "8) What motivates you when learning a new technology?",
         'secenekler': {
            '1': ("A new server-side framework/API technology", {'Backend': 3}),
            '2': ("A new UI library or design tool", {'Frontend': 3}),
            '3': ("A new test automation tool", {'QA/Test': 3}),
            '4': ("A new hardware/microcontroller platform", {'Embedded/IoT': 3}),
         }},
        {'soru': "9) Which question interests you most in a system?",
         'secenekler': {
            '1': ("How does this system scale to 1 million users?", {'DevOps/Cloud': 3}),
            '2': ("What kind of patterns can we extract from this dataset?", {'Data Science/AI-ML': 3}),
            '3': ("Does this app work properly on all devices?", {'QA/Test': 2, 'Mobile': 1}),
            '4': ("Is there a security vulnerability in this system?", {'Cybersecurity': 3}),
         }},
        {'soru': "10) What type of project appeals to you more?",
         'secenekler': {
            '1': ("Backend systems of an e-commerce site", {'Backend': 3}),
            '2': ("UI and mechanics of a mobile game", {'Game Development': 2, 'Mobile': 2}),
            '3': ("Software of smart home devices", {'Embedded/IoT': 3}),
            '4': ("Cloud management of a company's servers", {'DevOps/Cloud': 3}),
         }},
        {'soru': "11) What approach do you take when solving a problem?",
         'secenekler': {
            '1': ("Analyze data and make statistical inferences", {'Data Science/AI-ML': 3}),
            '2': ("List and test all possible error scenarios", {'QA/Test': 3}),
            '3': ("Think like an attacker and look for weak points", {'Cybersecurity': 3}),
            '4': ("Test the experience from the user's perspective", {'Frontend': 3}),
         }},
        {'soru': "12) Which technology field attracts you more?",
         'secenekler': {
            '1': ("Container/orchestration tools like Docker, Kubernetes", {'DevOps/Cloud': 3}),
            '2': ("UI libraries like React, Vue", {'Frontend': 3}),
            '3': ("Machine learning tools like TensorFlow, PyTorch", {'Data Science/AI-ML': 3}),
            '4': ("Hardware platforms like Arduino, Raspberry Pi", {'Embedded/IoT': 3}),
         }},
        {'soru': "13) In what role do you see yourself in the long term?",
         'secenekler': {
            '1': ("Someone who builds system architecture and manages the background", {'Backend': 3}),
            '2': ("Someone who shapes the user experience", {'Mobile': 3}),
            '3': ("Someone who builds decision support systems with data", {'Data Science/AI-ML': 3}),
            '4': ("Someone who ensures the security of systems", {'Cybersecurity': 3}),
         }},
        {'soru': "14) Which task is more enjoyable to you?",
         'secenekler': {
            '1': ("Doing performance tests for an application", {'QA/Test': 3}),
            '2': ("Coding the level design of a game", {'Game Development': 3}),
            '3': ("Writing code to process data coming from a sensor", {'Embedded/IoT': 3}),
            '4': ("Optimizing database queries of an API", {'Backend': 3}),
         }},
        {'soru': "15) What do you want to be remembered for most in your career?",
         'secenekler': {
            '1': ("A mobile app used by millions of users", {'Mobile': 3}),
            '2': ("An innovative AI product", {'Data Science/AI-ML': 3}),
            '3': ("A flawlessly tested, reliable system", {'QA/Test': 3}),
            '4': ("Large-scale, uninterrupted cloud infrastructure", {'DevOps/Cloud': 3}),
         }},
    ]

    alanlar = list(puanlar.keys())
    max_puan = {alan: 0 for alan in alanlar}
    for s in sorular:
        for alan in alanlar:
            en_iyi = max([secenek[1].get(alan, 0) for secenek in s['secenekler'].values()])
            max_puan[alan] += en_iyi

    for s in sorular:
        print(s['soru'])
        for numara, (metin, _) in s['secenekler'].items():
            print(f'   {numara}) {metin}')

        cevap = input('Your answer (1/2/3/4): ').strip()

        if cevap in s['secenekler']:
            _, secilen_puanlar = s['secenekler'][cevap]
            for alan, puan in secilen_puanlar.items():
                puanlar[alan] += puan
        else:
            print('Invalid answer, this question skipped.')
        print()

    yuzdeler = {alan: round((puanlar[alan] / max_puan[alan]) * 100, 1) for alan in alanlar}

    siralanmis = sorted(yuzdeler.items(), key=lambda x: x[1], reverse=True)
    onerilen_alan = siralanmis[0][0]

    print('📊 Match Percentage (high to low):')
    for alan, yuzde in siralanmis:
        print(f'   {alan}: %{yuzde}  (raw score: {puanlar[alan]}/{max_puan[alan]})')

    print(f"\n✅ Your most suitable software field: **{onerilen_alan}**")
    print(f'   (Second closest field: {siralanmis[1][0]})')
    print(f'\n🗺️  {onerilen_alan} recommended learning roadmap (roadmap.sh) for')
    print(f'   {roadmap_linkleri[onerilen_alan]}')

    return onerilen_alan, yuzdeler, roadmap_linkleri[onerilen_alan]


onerilen, tum_yuzdeler, roadmap_link = alan_oner()

## Phase 7: Export Model (for Deployment)

We save the trained salary prediction model to a file. This file can be loaded in a separate frontend/API project (e.g. FastAPI/Flask) to generate real-time predictions.

In [ ]:
# --- Save the final model (XGBoost pipeline) to disk ---
MODEL_DOSYA_ADI = 'maas_tahmin_modeli.joblib'

joblib.dump(model, MODEL_DOSYA_ADI)
print(f'Model saved: {MODEL_DOSYA_ADI}')

# --- Save the expected column order of the model (useful for deployment) ---
import json

model_bilgisi = {
    'ozellik_sirasi': list(X.columns),
    'kategorik_sutunlar': categorical_cols,
    'sayisal_sutunlar': ['WorkExp', 'DilSayisi'],
    'hedef_degisken': 'MonthlySalaryUSD (USD)',
    'usd_to_try_sabit_kur': USD_TO_TRY,
    'test_r2': round(r2_score(y_test, y_pred), 4),
    'test_mae': round(mean_absolute_error(y_test, y_pred), 2),
}

with open('model_bilgisi.json', 'w', encoding='utf-8') as f:
    json.dump(model_bilgisi, f, ensure_ascii=False, indent=2)

print('Model info saved: model_bilgisi.json')
print(json.dumps(model_bilgisi, ensure_ascii=False, indent=2))

**Deployment usage note:** The saved `maas_tahmin_modeli.joblib` file can be loaded in another Python environment (e.g. within a FastAPI/Flask API) using `joblib.load()` and called as `model.predict(new_data)`. `new_data` should be a `pandas.DataFrame` with the same order and names as the columns in the `ozellik_sirasi` list in `model_bilgisi.json`.

## Conclusion and Inference

This project combines an optimized XGBoost regression model that predicts the monthly salary from a developer's profile using the Stack Overflow Developer Survey 2025 data, with a fair (normalized) scoring system that recommends among 9 software fields based on personal preferences. Together, they provide both objective market data and personal inclination-based decision support for a software developer trying to determine their career direction.

## Checklist

- [x] Kaggle data read locally or auto-downloaded
- [x] Data size and columns verified (49,123 rows × 170 columns raw data)
- [x] Missing values and outliers cleaned
- [x] Numeric and categorical columns processed in a single model Pipeline
- [x] Train/test split applied (80%/20%)
- [x] 4 different models compared (LinearRegression, RandomForest, GradientBoosting, XGBoost)
- [x] XGBoost went through hyperparameter optimization with GridSearch
- [x] R² and MAE metrics calculated and interpreted
- [x] Model feature importance visualized
- [x] Working prediction function written for new profiles (USD + TL)
- [x] 9-field, 15-question, fair (normalized) scoring field recommender completed
- [x] Model exported for deployment (.joblib + .json)
- [x] Limitations and honest evaluation note added

## Next Steps

1. Port the model to an API with FastAPI/Flask and connect it to a frontend.
2. Make the fixed exchange rate assumption dynamic with a live currency API.
3. Connect the field recommender quiz to the model and combine them to automatically show a salary prediction based on the recommended field.
4. Periodically retrain the model with annual Stack Overflow surveys to keep the dataset updated.
5. Add README, installation instructions, and a live demo link to the GitHub repository.

**Final word:** This model does not guarantee a precise salary; it's a predictive tool trained on a limited set of features, systematically compared, optimized, and honestly evaluated. It should be used as a reference point, not standalone, when making career decisions.

---
*Software Persona Internship — 5-Day AI Training Capstone Project*